# 03d | Subqueries and CTEs

Complex SQL problems take several steps. In pandas you would use intermediate variables; in SQL the equivalents are **subqueries** and **CTEs** (`WITH` clauses).

**After this page you can:** write "compute A, then use A to compute B" as a single SQL query. About 45 minutes.

## 1. Subqueries: a query inside a query

"Employees paid more than the company average": first compute the average salary, then filter on it. You cannot write `AVG` directly in `WHERE`, but you can put a query there in parentheses:

In [ ]:
SELECT name, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees);

A subquery can also return a column, used together with `IN`:

In [ ]:
SELECT user_id, name
FROM users
WHERE user_id IN (SELECT user_id FROM orders WHERE amount >= 70);

### ✏️ Exercise 1 (write it yourself)

Find the **second-highest salary** (ties count as one value). Output a single column `second_salary`.

Idea: the highest of the salaries that are below the maximum.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

`WHERE salary < (SELECT MAX(salary) FROM employees)` removes the top salary; then take the `MAX` of what is left.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT MAX(salary) AS second_salary
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);
```

This is a classic. Another way to write it is `SELECT DISTINCT salary ... ORDER BY salary DESC LIMIT 1 OFFSET 1`; `DENSE_RANK` on the next page generalizes to "the Nth highest".

</details>

## 2. CTEs: naming intermediate results

Deeply nested subqueries are hard to read. A CTE writes out each step separately, gives it a name, and lets you use it afterwards like an ordinary table:

```sql
WITH name AS (
  a query
)
SELECT ... FROM name ...;
```

This is the same idea as writing `spend = orders.groupby(...)` in pandas and using `spend` on the next line.

Below, we first compute each user's completed spend (`spend`), then compute the average from `spend`:

In [ ]:
WITH spend AS (
  SELECT user_id, SUM(amount) AS total
  FROM orders
  WHERE status = 'completed'
  GROUP BY user_id
)
SELECT COUNT(*) AS n_buyers, AVG(total) AS avg_total, MAX(total) AS max_total
FROM spend;

Separate multiple CTEs with commas; later ones can use earlier ones:

```sql
WITH a AS (...),
     b AS (SELECT ... FROM a ...)
SELECT ... FROM b;
```

**Interview tip:** use CTEs for any problem with more than two steps, have each CTE do one thing, and give them meaningful names. It is easier for the interviewer to read and harder for you to get lost.

### ✏️ Exercise 2 (fill in the blanks)

Find users whose completed spend is **above average** (the average over all users who have completed orders). Output `user_id`, `total`.

In [ ]:
WITH spend AS (
  SELECT user_id, SUM(amount) AS total
  FROM orders
  WHERE status = 'completed'
  GROUP BY user_id
)
SELECT user_id, total
FROM ____
WHERE total > (SELECT ____(total) FROM ____);

<details>
<summary><b>Answer</b></summary>

```sql
WITH spend AS (
  SELECT user_id, SUM(amount) AS total
  FROM orders
  WHERE status = 'completed'
  GROUP BY user_id
)
SELECT user_id, total
FROM spend
WHERE total > (SELECT AVG(total) FROM spend);
```

</details>

### ✏️ Exercise 3 (write it yourself)

Find employees paid **more than their department's average salary**. Output `name`, `dept`, `salary`.

Steps: compute each department's average salary in a CTE, then join it to `employees`.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

CTE: `SELECT dept, AVG(salary) AS avg_salary FROM employees GROUP BY dept`. Main query: `employees e JOIN dept_avg d ON e.dept = d.dept WHERE e.salary > d.avg_salary`.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
WITH dept_avg AS (
  SELECT dept, AVG(salary) AS avg_salary
  FROM employees
  GROUP BY dept
)
SELECT e.name, e.dept, e.salary
FROM employees e
JOIN dept_avg d
  ON e.dept = d.dept
WHERE e.salary > d.avg_salary;
```

</details>

### ✏️ Exercise 4 (write it yourself)

**Conversion rate by channel** (same as Problem 1 in 02f, this time in SQL).

A user "converts" if they have at least one `completed` order. For each `channel`, output `channel`, `n_users`, `n_converted`, `rate`.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

① CTE `buyers`: the distinct `user_id`s with a completed order (`DISTINCT` guarantees one row per user). ② `users LEFT JOIN buyers`. ③ Group by channel: `COUNT(*)` is the number of users, `COUNT(b.user_id)` the number converted (non-converters are NULL and not counted). ④ Remember to multiply by `1.0` for the ratio.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
WITH buyers AS (
  SELECT DISTINCT user_id
  FROM orders
  WHERE status = 'completed'
)
SELECT u.channel,
       COUNT(*)                        AS n_users,
       COUNT(b.user_id)                AS n_converted,
       1.0 * COUNT(b.user_id) / COUNT(*) AS rate
FROM users u
LEFT JOIN buyers b
  ON u.user_id = b.user_id
GROUP BY u.channel;
```

</details>

> Note the `DISTINCT` in `buyers`: each user matches at most one row, so after the left join there is still one row per user and `COUNT(*)` equals the number of users. This is the "aggregate to one row per unit first, then join" rule from 02d.

## 3. UNION

`UNION ALL` stacks the results of two queries (pandas' `concat`); `UNION` also removes duplicates afterwards. Both queries must have the same number and types of columns.

In [ ]:
SELECT name, 'user' AS kind FROM users WHERE country = 'CA'
UNION ALL
SELECT name, 'employee' AS kind FROM employees WHERE dept = 'Mkt';

### ✏️ Exercise 5 (write it yourself)

Find each user's **first order** (the earliest date). Output `user_id`, `order_id`, `order_date`.

Steps: compute each user's earliest order date in a CTE, then join back to `orders` to get that row.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

There are two join conditions: `o.user_id = f.user_id AND o.order_date = f.first_date`.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
WITH first AS (
  SELECT user_id, MIN(order_date) AS first_date
  FROM orders
  GROUP BY user_id
)
SELECT o.user_id, o.order_id, o.order_date
FROM orders o
JOIN first f
  ON o.user_id = f.user_id
 AND o.order_date = f.first_date;
```

This version avoids window functions. The next page uses `ROW_NUMBER` to solve this kind of "one row per group" problem more concisely.

</details>

## Summary

- Subqueries can go in `WHERE` (returning a single value or a column) and in `FROM`;
- For multi-step problems use `WITH name AS (...)`, one CTE per step;
- Before joining, reshape the right side to "one row per unit".

On to **03e**.